# Brasa — pré-treino do zero (GPU do Colab)
Treina um modelo **próprio, sem pesos externos**, com `python/model.py` do projeto. Escolha *Ambiente de execução → Alterar tipo → GPU* (A100/L4 ideal; T4 serve para o preset `small`).
Tudo é salvo no Drive e **retomável**: se a sessão cair, rode as células de novo.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/brasa-pretrain'
!mkdir -p {WORK}

In [ ]:
# Repositório privado? Gere um token (GitHub > Settings > Developer settings) e preencha.
GITHUB_TOKEN = ''
import os
url = 'github.com/va415988-glitch/Brasa-IA.git'
auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
if not os.path.exists('/content/Brasa-IA'):
    !git clone https://{auth}{url} /content/Brasa-IA
else:
    !git -C /content/Brasa-IA pull
%cd /content/Brasa-IA
!pip -q install datasets tokenizers numpy

In [ ]:
# 1) Dados abertos + tokenizer próprio. Demora (download). SCALE=1 ~ 7 GB de texto (~1,5 bi de tokens).
# Se {WORK}/data já existe (rodada anterior), tudo é verificado e pulado em poucos minutos.
SCALE = 1.0
!python pretrain/prepare_data.py --out {WORK}/data --scale {SCALE}

In [ ]:
# 2) Treino do zero. PRESET: 'base' (~100M, A100: ~4-5 h para 2 bi de tokens), 'small' (~40M, T4) ou 'large' (~300M, precisa de mais dados).
# Falta de memória? Reduza MICRO (L4: 8). MAX_HOURS para o treino com segurança antes da sessão acabar; rode de novo para retomar.
# A receita atual (Muon + WSD + QK-norm) não retoma estados da receita antiga: por isso a pasta leva RUN.
PRESET, MICRO, MAX_HOURS, RUN = 'base', 16, 10, 'v2'
!python pretrain/train.py --data {WORK}/data --out {WORK}/out-{PRESET}-{RUN} --preset {PRESET} --micro-batch {MICRO} --max-hours {MAX_HOURS} --compile

Terminou ou parou por tempo? Rode a célula 2 de novo: ela retoma de `last.pt`.
Pesos finais: `brasa-pretrain/out-<preset>-<run>/best.safetensors` (+ `.json`) e o tokenizer em `brasa-pretrain/data/tokenizer.json`. Baixe-os para `model/pretrained/` no seu computador.

### 2b) Continuar um modelo já treinado (alternativa à célula 2)
O `small` de 02/10 viu só ~52% dos dados (784 M de 1,5 bi de tokens) e já decaiu o LR. `--init` parte dos pesos dele
(arquitetura herdada, sem QK-norm) com Muon + WSD novos. Num teste A/B pequeno, continuar assim deu perda menor que retomar com AdamW + cosseno.

In [ ]:
PRESET, MICRO, MAX_HOURS, RUN = 'small', 16, 10, 'cont'
TOKENS = 1.5e9  # ~1 época a mais; aumente se o val_loss ainda estiver caindo
!python pretrain/train.py --data {WORK}/data --init {WORK}/out-small/best.safetensors --out {WORK}/out-small-{RUN} --tokens {TOKENS} --micro-batch {MICRO} --max-hours {MAX_HOURS} --compile
# O SFT abaixo usa {WORK}/out-{PRESET}-{RUN}/best.safetensors.

## 3) SFT — conversar, agir como agente e responder no formato do produto
Mistura, com a origem de cada parte registrada no checkpoint:
- **decisões de agente** (`datasets/agent_sft_v1`): o protocolo de `cognitive_dialogue.py` com ferramentas reais do runtime sobre 105 repositórios abertos, pesquisa com artigos reais da Wikipédia, respostas humanas (OpenAssistant 2, Dolly) e diálogos escritos por LLM (rotulados `authored-llm-v1`);
- **conversa**: OpenAssistant 2 e Dolly completos, conversas do projeto e os diálogos escritos, como chat;
- **código**: problemas do MBPP no contrato de plano JSON da bancada;
- **disciplina de evidência**: `cognitive_sft_v2` e `cognitive_alignment_v4`.
`--replay 0.2` mistura texto de pré-treino para reduzir o esquecimento (no `small`, o SFT piorou a perplexidade geral de 41,5 para 56,2).

In [ ]:
BASE = f'{WORK}/out-{PRESET}-{RUN}/best.safetensors'  # o modelo da célula 2 ou 2b, a que você rodou por último
!python pretrain/build_code_sft.py --out {WORK}/data/sft_code_plans.jsonl
LOCAL = ','.join(['python/data/*.jsonl', f'{WORK}/data/sft_code_plans.jsonl',
                  'datasets/agent_sft_v1/train.jsonl.gz', 'datasets/agent_sft_v1/authored_chat.jsonl.gz',
                  'datasets/cognitive_sft_v2/train.jsonl', 'datasets/cognitive_alignment_v4/train.jsonl'])
!python pretrain/sft.py --data {WORK}/data --base {BASE} --out {WORK}/sft-{PRESET}-{RUN} --hf oasst2,dolly --epochs 3 --replay 0.2 --local "{LOCAL}"

## 4) Baixar e medir no computador
Baixe `sft-<preset>-<run>/sft.safetensors` (+ `.json`) e `data/tokenizer.json`. No computador, copie o tokenizer para
`model/pretrained/tokenizer.json` (caminho que o checkpoint declara) e rode as duas bancadas (o servidor roda em CPU; no Colab é lento demais):

- agente: `python pretrain/eval_agent_sft.py --checkpoint model/pretrained/sft.safetensors --report <relatorio-agente>.json`
- programação: `python scripts/evaluate_programming_qualification.py --suite reserved --backend checkpoint --checkpoint model/pretrained/sft.safetensors --output <novo-relatorio>.json`

Referências (02-03/10, `small`): agente 0% de decisões válidas antes do treino agêntico; programação 0/96 aprovadas, 22 no contrato JSON.